In [1]:
import os
import torch
import torchvision.transforms as transforms
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder
import timm
from torchvision.transforms import functional as F
import qoi
from PIL import Image
from tqdm import tqdm

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

# 🔹 QOI 파일 캐시
qoi_cache = {}

import numpy as np

def mixup_data(x, y, alpha=1.0):
    '''MixUp 이미지 및 라벨 생성'''
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0

    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    '''MixUp용 Loss'''
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def qoi_loader(qoi_path):
    if qoi_path in qoi_cache:
        return qoi_cache[qoi_path]

    if not os.path.exists(qoi_path):
        print(f"🚨 파일 없음: {qoi_path}")
        return None

    with open(qoi_path, "rb") as f:
        qoi_data = f.read()

    img = qoi.decode(qoi_data)
    img = Image.fromarray(img).convert("RGB")  # RGB 변환

    qoi_cache[qoi_path] = img  # 캐시에 저장
    return img

# 🔹 데이터 전처리
class ResizeWithPadding:
    def __init__(self, size=224, padding_color=(0, 0, 0)):
        self.size = size
        self.padding_color = padding_color

    def __call__(self, img):
        w, h = img.size
        scale = self.size / max(w, h)
        new_w, new_h = int(w * scale), int(h * scale)
        img = F.resize(img, (new_h, new_w))

        delta_w = self.size - new_w
        delta_h = self.size - new_h
        padding = (delta_w // 2, delta_h // 2, delta_w - delta_w // 2, delta_h - delta_h // 2)

        return F.pad(img, padding, fill=self.padding_color, padding_mode="constant")

# 🔹 데이터 변환
transform_train = transforms.Compose([
    ResizeWithPadding(size=224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

transform_test = transforms.Compose([
    ResizeWithPadding(size=224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 🔹 데이터셋 로드
train_path = "/root/Public_Storage/madelab_khw/lpcv/coco/cropped_dir_train"
test_path = "/root/Public_Storage/madelab_khw/lpcv/coco/cropped_dir_test"

dataset = ImageFolder(root=train_path, transform=transform_train, loader=qoi_loader, is_valid_file=lambda path: path.endswith(".qoi"))
test_dataset = ImageFolder(root=test_path, transform=transform_test, loader=qoi_loader, is_valid_file=lambda path: path.endswith(".qoi"))

# 🔹 훈련 및 검증 데이터셋 분할
train_size = int(0.8 * len(dataset))
valid_size = len(dataset) - train_size
train_dataset, valid_dataset = random_split(dataset, [train_size, valid_size])

# 🔹 배치 크기 및 데이터 로더 설정
batch_size = 128  # 기존 256 → 128로 감소
num_workers = 0  # 데이터 로딩 속도 최적화

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

# 🔹 모델 생성 및 DataParallel 적용
num_classes = 64  # 원하는 클래스 개수
model = timm.create_model("convnext_tiny", pretrained=True, num_classes=64)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.device_count() > 1:
    print(f"🔹 {torch.cuda.device_count()} 개의 GPU 사용 중")
    model = torch.nn.DataParallel(model)  # DataParallel 적용

model = model.to(device)

# 🔹 CUDA 최적화
torch.backends.cudnn.benchmark = True  # GPU 연산 속도 최적화

# 🔹 손실 함수 및 최적화 설정
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# 🔹 학습 루프
num_epochs = 10
best_valid_acc = 0.0
save_path = "/root/Public_Storage/madelab_khw/lpcv/model/convnext_tiny.pth"

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", unit="batch")

    mixup_prob = 0.5  # MixUp을 적용할 확률

    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)

        if np.random.rand() < mixup_prob:
            # ✅ 확률적으로 MixUp 적용
            images, targets_a, targets_b, lam = mixup_data(images, labels, alpha=0.4)
            outputs = model(images)
            loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)

            _, predicted = outputs.max(1)
            correct += predicted.eq(targets_a).sum().item()  # 참고용
        else:
            # ✅ MixUp 없이 원본 그대로 학습
            outputs = model(images)
            loss = criterion(outputs, labels)

            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        total += labels.size(0)
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")


    train_acc = 100 * correct / total
    scheduler.step()

    # 🔹 검증 루프
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

    valid_acc = 100 * correct / total

    # 🔹 최고 성능 모델 저장
    if valid_acc > best_valid_acc:
        best_valid_acc = valid_acc
        torch.save(model.module.state_dict() if torch.cuda.device_count() > 1 else model.state_dict(), save_path)

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss:.4f}, Train Acc: {train_acc:.2f}%, Valid Acc: {valid_acc:.2f}%")

print("✅ 학습 완료!")

# 🔹 최적 모델 로드
# model.load_state_dict(torch.load(save_path))
# model.eval()

# # 🔹 테스트 평가
# correct = 0
# total = 0

# with torch.no_grad():
#     for images, labels in tqdm(test_loader, desc="Testing", unit="batch"):
#         images, labels = images.to(device), labels.to(device)

#         outputs = model(images)
#         _, predicted = outputs.max(1)
#         correct += predicted.eq(labels).sum().item()
#         total += labels.size(0)

# test_acc = 100 * correct / total
# print(f"✅ EfficientNet-B4 최종 테스트 정확도: {test_acc:.2f}%")


🔹 2 개의 GPU 사용 중


Epoch 1/10:   0%|                                                                                           | 0/1433 [00:00<?, ?batch/s]

Epoch 1/10:   0%|                                                                              | 0/1433 [00:07<?, ?batch/s, loss=4.3857]

Epoch 1/10:   0%|                                                                    | 1/1433 [00:07<2:49:13,  7.09s/batch, loss=4.3857]

Epoch 1/10:   0%|                                                                    | 1/1433 [00:09<2:49:13,  7.09s/batch, loss=4.2932]

...

Epoch 4/10:  99%|██████████████████████████████████████████████████████████████████▏| 1415/1433 [41:11<00:31,  1.74s/batch, loss=1.6065]

Epoch 4/10:  99%|██████████████████████████████████████████████████████████████████▏| 1416/1433 [41:11<00:29,  1.76s/batch, loss=1.6065]

Epoch 4/10:  99%|██████████████████████████████████████████████████████████████████▏| 1416/1433 [41:12<00:29,  1.76s/batch, loss=0.8546]

Epoch 4/10:  99%|██████████████████████████████████████████████████████████████████▎| 1417/1433 [41:12<00:27,  1.73s/batch, loss=0.8546]

Epoch 4/10:  99%|██████████████████████████████████████████████████████████████████▎| 1417/1433 [41:14<00:27,  1.73s/batch, loss=0.9105]